# Aula 04 — Processamento de Linguagem Natural

Células na ordem dos slides; algumas dependem de variáveis de células anteriores.

## Slide 8 — EMBEDDINGS DENSOS POUCOS NÚMEROS, MUITA SEMÂNTICA

*Bloco 01*

In [23]:
# Em embeddings (300 dimensões)
vec_rei      = [0.42, -0.13, 0.78, 0.21, ..., -0.05]   # 300 floats
vec_rainha   = [0.41, -0.18, 0.75, 0.19, ..., -0.07]   # 300 floats
vec_abacaxi  = [-0.31, 0.62, 0.04, -0.55, ..., 0.88]   # 300 floats
# Similaridade do cosseno:
# cos_sim(rei, rainha)  = 0.72   # ALTAMENTE similares!
# cos_sim(rei, abacaxi) = 0.05   # praticamente sem relação
# Espaço vetorial onde GEOMETRIA = SEMÂNTICA

## Dados de exemplo (adicionado no notebook)

O slide seguinte usa `reviews`, que não aparece nos slides. Usamos reviews sintéticas só para o Word2Vec rodar.

In [24]:
# [notebook] Dados sintÃ©ticos de exemplo â€” os slides assumem um corpus "jÃ¡ preparado" (reviews + labels)
import random
random.seed(42)
_pos = ["Adorei o produto", "Excelente qualidade", "Entrega rÃ¡pida e bem embalado", "Recomendo muito",
        "Funciona perfeitamente", "Ã“timo atendimento", "Superou minhas expectativas", "Produto maravilhoso"]
_neg = ["PÃ©ssimo produto", "Chegou quebrado", "Entrega atrasada", "NÃ£o recomendo",
        "Veio com defeito", "Atendimento horrÃ­vel", "Qualidade ruim", "Muito decepcionado"]
_neu = ["amei esse item", "ganhei com amor", "comprei na semana passada", "o preÃ§o estava bom", "a embalagem era simples", "usei por alguns dias",
        "chegou em casa ontem", "a cor Ã© como na foto"]
reviews_corpus, labels = [], []
for _ in range(400):
    y_ = random.randint(0, 1)
    frases = random.sample(_pos if y_ else _neg, 2) + random.sample(_neu, 1)
    random.shuffle(frases)
    reviews_corpus.append(". ".join(frases) + ".")
    labels.append(y_)
reviews = reviews_corpus  # nome usado na aula 05
print(len(reviews_corpus), "reviews | positivas:", sum(labels))

400 reviews | positivas: 205


## Slide 17 — TREINANDO WORD2VEC COM GENSIM SETUP E TREINAMENTO EM PT-BR

*Bloco 02*

In [28]:
import nltk

nltk.download('punkt_tab')

%pip install gensim nltk
%pip install gensim
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt', quiet=True)
# Hipotético: 'reviews' é uma lista de strings em PT
# Tokenizamos cada review (já com pré-processamento da Aula 2)
sentences = [word_tokenize(r.lower(), language='portuguese')
             for r in reviews]
# Treina Word2Vec com Skip-gram + negative sampling
model = Word2Vec(
    sentences=sentences,
    vector_size=100,        # dimensionalidade do embedding
    window=5,               # janela de contexto
    min_count=5,            # ignora palavras com freq < 5
    workers=4,              # paralelismo
    sg=1,                   # 1 = Skip-gram, 0 = CBOW
    negative=10,            # negative sampling (10 amostras)
    epochs=10,
)
model.save('word2vec_ptbr.model')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


## Slide 18 — USANDO O MODELO TREINADO ACESSANDO VETORES E SIMILARIDADES

*Bloco 03*

In [29]:
# Vetor de uma palavra
print(model.wv['produto'][:5])
# Saída: array([0.123, -0.456, 0.789, 0.234, -0.567])
# Tamanho do vetor (vector_size=100)
print(model.wv['produto'].shape)
# Saída: (100,)
# Palavras mais similares
print(model.wv.most_similar('produto', topn=5))
# Saída:
# [('item', 0.78),
#  ('aparelho', 0.74),
#  ('mercadoria', 0.69),
#  ('artigo', 0.65),
#  ('peça', 0.61)]
# Similaridade entre duas palavras
print(model.wv.similarity('produto', 'item'))     # 0.78
print(model.wv.similarity('produto', 'amor'))     # 0.12
# Carregar modelo salvo
model = Word2Vec.load('word2vec_ptbr.model')

[-0.05140453  0.15142754 -0.01220048  0.09486274 -0.04513614]
(100,)
[('adorei', 0.9969369769096375), ('maravilhoso', 0.995836615562439), ('perfeitamente', 0.9955404996871948), ('amor', 0.9943480491638184), ('.', 0.9942652583122253)]
0.9870934
0.99434817


> ⚠️ **Atenção (adicionado no notebook):** a partir daqui as células de FastText usam o modelo oficial `wiki.pt.bin` (~7 GB, ~7 GB de RAM). Baixe antes de rodar:
> `https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.pt.bin` e salve na mesma pasta do notebook. Sem o arquivo, essas células dão `FileNotFoundError`. O slide 38 (fine-tuning) é ilustrativo: precisa de `nilc-w2v-skip-300.txt` e de um `meu_corpus_dominio` seu.

## Slide 31 — FASTTEXT RESOLVE OOV NÃO HÁ MAIS "PALAVRA DESCONHECIDA"

*Bloco 04*

In [31]:
%pip install gensim
from gensim.models.fasttext import load_facebook_vectors
# Modelo FastText oficial em português
modelo = load_facebook_vectors('wiki.pt.bin')
# Vetor de palavra comum
v = modelo['amor']
print(v.shape)  # (300,)
# Vetor de NEOLOGISMO que NUNCA esteve no corpus
v_novo = modelo['instagramável']
print(v_novo.shape)  # (300,) — foi inferido pelos subwords!
# Vetor de gíria
v_giria = modelo['baita']
print(v_giria.shape)  # (300,) — funciona
# Em Word2Vec, isso jogaria KeyError. Em FastText, sempre tem vetor.

AttributeError: module 'gensim' has no attribute 'download'

## Slide 36 — FASTTEXT OFICIAL (FACEBOOK) 157 LÍNGUAS, INCLUINDO PT

*Bloco 05*

In [ ]:
# Download do modelo PT (~7 GB)
# wget https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.pt.bin
# Carregando com Gensim
from gensim.models.fasttext import load_facebook_vectors
modelo = load_facebook_vectors('wiki.pt.bin')
print(modelo['celular'].shape)   # (300,)
print(modelo.most_similar('celular', topn=5))
# [('smartphone', 0.81), ('telefone', 0.78),
#  ('aparelho', 0.71), ('android', 0.68), ('iphone', 0.65)]
# Versão menor (.vec) só com vetores principais ~1.5 GB
# https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.pt.vec

## Slide 38 — QUANDO VALE FINE-TUNING? ADAPTANDO PRÉ-TREINADOS AO SEU DOMÍNIO

*Bloco 06*

In [ ]:
from gensim.models import Word2Vec, KeyedVectors
# Carrega embeddings pré-treinados
base = KeyedVectors.load_word2vec_format('nilc-w2v-skip-300.txt')
# Inicia novo modelo com os pesos pré-treinados
model = Word2Vec(vector_size=300, min_count=5, sg=1)
model.build_vocab(meu_corpus_dominio)
model.wv.vectors = np.copy(base.vectors)  # transfere pesos
# Continua o treinamento no SEU corpus (fine-tuning)
model.train(meu_corpus_dominio, total_examples=len(meu_corpus_dominio),
            epochs=5)

## Slide 47 — CARREGANDO EMBEDDINGS PRÉ-TREINADOS FASTTEXT OFICIAL EM PT-BR

*Bloco 07*

In [ ]:
from gensim.models.fasttext import load_facebook_vectors
# Baixe antes:
# wget https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.pt.bin
# Carregando o modelo (pode demorar ~30s, ocupa ~7GB de RAM)
modelo = load_facebook_vectors('wiki.pt.bin')
# Vetor de uma palavra
v = modelo['amor']
print(v.shape)  # (300,)
print(v[:5])    # array([0.123, -0.456, 0.789, 0.234, -0.567])
# Funciona até com OOV graças a subwords:
v_neologismo = modelo['instagramável']
print(v_neologismo.shape)  # (300,) - foi inferido pelos subwords
# Tamanho do vocabulário
print(f'Vocabulário: {len(modelo.key_to_index):,} palavras')
# ~2 milhões de palavras conhecidas + qualquer OOV via subword

## Slide 49 — SIMILARIDADE DO COSSENO EM CÓDIGO COM GENSIM

*Bloco 08*

In [ ]:
# Similaridade entre duas palavras
print(modelo.similarity('rei', 'rainha'))      # ~0.72
print(modelo.similarity('rei', 'abacaxi'))     # ~0.05
print(modelo.similarity('feliz', 'alegre'))    # ~0.78
print(modelo.similarity('feliz', 'triste'))    # ~0.45 (relacionados, opostos!)
# Palavras mais similares (top-k)
print(modelo.most_similar('celular', topn=5))
# [('smartphone', 0.81),
#  ('telefone', 0.78),
#  ('aparelho', 0.71),
#  ('android', 0.68),
#  ('iphone', 0.65)]
# Em PT-BR
print(modelo.most_similar('feliz', topn=5))
# [('alegre', 0.78), ('contente', 0.74), ('eufórico', 0.71),
#  ('animado', 0.68), ('satisfeito', 0.65)]

## Slide 51 — ANALOGIAS EM CÓDIGO "REI ESTÁ PARA HOMEM COMO X ESTÁ PARA MULHER"

*Bloco 09*

In [ ]:
# A famosa analogia: rei está para homem assim como X está para mulher
# Aritmética: vec(rei) - vec(homem) + vec(mulher) = ?
resultado = modelo.most_similar(
    positive=['rei', 'mulher'],   # adiciona
    negative=['homem'],            # subtrai
    topn=3
)
print(resultado)
# [('rainha', 0.78),
#  ('princesa', 0.72),
#  ('imperatriz', 0.68)]
# Por que funciona?
# A direção homem→mulher no espaço vetorial codifica a relação masculino→feminino
# Aplicada a 'rei', leva ao equivalente feminino: 'rainha'
# Outras analogias:
modelo.most_similar(
    positive=['paris', 'alemanha'],
    negative=['frança'],
    topn=3
)
# [('berlim', 0.74), ('hamburgo', 0.62), ('munique', 0.58)]

## Slide 56 — VISUALIZAÇÃO EM CÓDIGO CLUSTERS SEMÂNTICOS COM T-SNE

*Bloco 10*

In [32]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
# Selecionamos um conjunto de palavras de interesse
palavras = ['cachorro', 'gato', 'leão', 'tigre', 'elefante',
            'carro', 'moto', 'ônibus', 'caminhão', 'bicicleta',
            'feliz', 'triste', 'alegre', 'deprimido', 'eufórico']
# Cada palavra tem vetor de 300 dimensões
vetores = np.array([modelo[p] for p in palavras])
print(vetores.shape)  # (15, 300)
# Reduz para 2D com t-SNE
tsne = TSNE(n_components=2, perplexity=5, random_state=42)
vetores_2d = tsne.fit_transform(vetores)
print(vetores_2d.shape)  # (15, 2)
# Plota
plt.figure(figsize=(10, 8))
plt.scatter(vetores_2d[:, 0], vetores_2d[:, 1])
for i, palavra in enumerate(palavras):
    plt.annotate(palavra, (vetores_2d[i, 0], vetores_2d[i, 1]))
plt.title('Clusters semânticos em PT-BR (FastText)')
plt.show()

NameError: name 'modelo' is not defined